In [ ]:
import scanpy as sc
import anndata as ad
import scanpy.external as sce

import pandas as pd
import numpy as np

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

import gseapy as gp

import matplotlib.pyplot as plt
import seaborn as sns

from sccoda.util import comp_ana as mod
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz
import sccoda.datasets as scd
import arviz as az

import scvi
import torch
import scipy.sparse as sp

from rich import print
import warnings
warnings.filterwarnings("ignore")
import os

outdir = "/scratch/user/s4575250/BIOX7018_thesis/write/03_sccoda_DA/PICA0001-PICA0007"
os.makedirs(outdir, exist_ok=True)
sc.settings.figdir = "/scratch/user/s4575250/BIOX7018_thesis/figures/PICA0001-PICA0007/03_sccoda"

### Abundance analysis (sccoda)

In [ ]:
adata_age_nk= sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_nk_combined_annot_with_age_scvi.h5ad")

In [ ]:
# Setup cell type by donor table
ct_table = (
    adata_age_nk.obs.groupby(["pica_id", "cell_type"])
    .size()
    .unstack(fill_value=0)
)

print("ct_table shape:", ct_table.shape)
print(ct_table)

ct_table shape:
(7, 12)

cell_type  CCR6+ memory CD4 T cell  Cytotoxic CD4 T cell  \
pica_id                                                    
PICA0001                        22                    64   
PICA0002                       176                     2   
PICA0003                        18                     2   
PICA0004                        20                     6   
PICA0005                       125                     6   
PICA0006                        14                     0   
PICA0007                        68                     6   

cell_type  Cytotoxic natural killer cell  Efector memory CD8 T cell  \
pica_id                                                               
PICA0001                             446                         23   
PICA0002                             748                         75   
PICA0003                             398                         44   
PICA0004                             811                         19   
PICA0005                            2412                         36   
PICA0006                             662                         61   
PICA0007                             487                         70   

cell_type  GZMK+ memory CD8 T cell  Gamma-delta T cell  NKT cell  \
pica_id                                                            
PICA0001                        11                  96         2   
PICA0002                        91                  88        15   
PICA0003                        43                  73         6   
PICA0004                         8                  67         2   
PICA0005                        52                 513         6   
PICA0006                        24                  57         5   
PICA0007                        40                 215        19   

cell_type  Naive/Central memory CD4 T cell  Naive/Central memory CD8 T cell  \
pica_id                                                                       
PICA0001                              4436                              520   
PICA0002                              1625                              349   
PICA0003                              2203                              386   
PICA0004                              2163                              305   
PICA0005                              1060                              306   
PICA0006                              2953                              651   
PICA0007                              3192                              716   

cell_type  Regulatory T cell  Suppressed CD8 T cell  Tfh / cTfh  
pica_id                                                          
PICA0001                  74                    229         210  
PICA0002                  52                     54         176  
PICA0003                  76                     67          89  
PICA0004                  74                    111         155  
PICA0005                  16                    237         162  
PICA0006                 128                      9         109  
PICA0007                 114                    141         367

## Creating scCODA object for larger datasets
- using Age_group instead of Age in years int

In [ ]:
meta_df = (
    adata_age_nk.obs[["pica_id", "Age_group"]]
    .drop_duplicates()
    .set_index("pica_id")
)

meta_df

In [ ]:
# merge donor metadata into the counts table
ct_table_with_meta = ct_table.merge(meta_df, left_index=True, right_index=True)

# create scCODA data object
sccoda_data = dat.from_pandas(ct_table_with_meta, ["Age_group"])

print(sccoda_data)

In [ ]:
# check input to scCODA
viz.boxplots(sccoda_data, feature_name="Age_group")
plt.show()

In [ ]:
# run scCODA compositional analysis
model = mod.CompositionalAnalysis(
    formula="Age_group",                            
    data=sccoda_data,
    reference_cell_type="type here"  
)
age_results = model.sample_hmc(20000, 4)

In [ ]:
age_results.summary()

In [ ]:
# credible effect summary
credible=age_results.credible_effects().reset_index()
print(credible)

In [ ]:
intercepts_df = age_results.intercept_df.reset_index()
effects_df = age_results.effect_df.reset_index()

intercepts_df.to_csv("/scratch/user/s4575250/BIOX7018_thesis/figures/PICA/03_sccoda/nk_age_sccoda_intercepts.csv", index=False)
effects_df.to_csv("/scratch/user/s4575250/BIOX7018_thesis/figures/PICA/03_sccoda/nk_age_sccoda_effects.csv", index=False)
credible.to_csv("/scratch/user/s4575250/BIOX7018_thesis/figures/PICA/03_sccoda/nk_age_sccoda_credible_effects.csv", index=False)

In [ ]:
# extract coefficients summary dataframe
summary_df = az.summary(age_results)
print(summary_df.head())

In [ ]:
# subset regression coefficients
coef_df = summary_df[summary_df.index.str.contains("beta", case=False)].copy()
print(coef_df.head())

In [ ]:
# parse cell type from index and sort by mean coefficient
coef_df = coef_df.reset_index()
coef_df["cell_type"] = coef_df["index"].str.extract(r", (.*?)\]")
coef_df = coef_df.sort_values("mean", ascending=False)
print(coef_df.head())

In [ ]:
print(coef_df.columns)

In [ ]:
coef_df.rename(columns={"hdi_3%": "hdi_3", "hdi_97%": "hdi_97"}, inplace=True)

In [ ]:
# forest plot of compositional analysis results
plt.figure(figsize=(8,6))

sns.pointplot(
    data=coef_df,
    x="mean",
    y="cell_type",
    join=False,
    color="steelblue"
)

# add credible intervals (HDI)
for i, row in enumerate(coef_df.itertuples()):
    plt.plot([row.hdi_3, row.hdi_97], [i, i], color="black", lw=1)

plt.axvline(0, color="red", linestyle="--")
plt.xlabel("Log-fold change (Age)")
plt.ylabel("Cell subset")
plt.title("scCODA compositional changes with age")
plt.tight_layout()
plt.show()


In [ ]:
# inclusion probability plot
posterior = age_results.posterior
beta_vars = [v for v in posterior.data_vars if "beta" in v]

for var in beta_vars:
    mean_val = posterior[var].mean().values
    hdi = az.hdi(posterior[var])
    print(f"{var}: mean={mean_val}, HDI={hdi}")
    
plt.savefig("/scratch/user/s4575250/BIOX7018_thesis/figures/PICA/03_sccoda/PICA_nk_sccoda_prob_plot.png", dpi=300)


In [ ]:
# Convert raw cell counts to proportions per donor
prop_df = ct_table_with_meta.div(ct_table_with_meta.sum(axis=1), axis=0)
prop_df["Age_group"] = meta_df["Age_group"]

# Melt for plotting
prop_melt = prop_df.reset_index().melt(
    id_vars=["pica_id", "Age_group"],
    var_name="Cell type",
    value_name="Proportion"
)

plt.figure(figsize=(10,4))
sns.boxplot(
    data=prop_melt,
    x="Cell type",
    y="Proportion",
    hue="Age_group",
    palette="Blues"
)
plt.title("Donor-level cell-type proportions by age")
plt.xticks(rotation=45)
plt.tight_layout()    
plt.savefig("/scratch/user/s4575250/BIOX7018_thesis/figures/PICA/03_sccoda/PICA_nk_sccoda_raw_donor_level_prop.png", dpi=300)
plt.show()